In [4]:
import pandas as pd
import numpy as np
import os

In [5]:
path = "dataset"

In [6]:
os.chdir(path)

FileNotFoundError: [WinError 2] Sistem belirtilen dosyayı bulamıyor: 'dataset'

In [ ]:
df = pd.read_csv('train.csv', sep = ',')

In [ ]:
df.head(10)

In [ ]:
df['diagnosis'].hist()
df['diagnosis'].value_counts()

Retinopati görüntülerini aktarmak için, **os.listdir()** kullanarak *''train_img''* klasöründe bulunan dosyaların isimlerini *''files''* ismine atadım.
* '000c1434d8d7.png'
* '001639a390f0.png'
* '0024cdab0c1e.png'
*  ...

In [ ]:
import os
files = os.listdir('train_img')
# files

In [ ]:
len(files)

In [ ]:
import cv2

OpenCV'nin [görüntü okuma modülü](https://opencv-python-tutroals.readthedocs.io/en/latest/py_tutorials/py_gui/py_image_display/py_image_display.html)nü kullanarak **files**'ın içindeki görüntü isimleri üzerinden bir for döngüsü döndürerek görüntüleri okuduk. Ardından bunları (400,400,3) boyutunda yeniden şekillendirdik. OpenCV görüntüleri BGR şeklinde okuduğu için bunları RGB renk koduna çevirdik. Ardından döngüde dönen her görüntüyü *img_list* ismindeki listeye ekledik.

In [ ]:
img_list = []

for i in files[0:20]:
    image = cv2.imread(path + 'train_img\\'+i)
    image = cv2.resize(image,(400,400))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_list.append(image)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.imshow(img_list[0])

In [ ]:
plt.imshow(img_list[4])

In [ ]:
kopya = img_list[4].copy()

#### Threshold uygulayabilmek için görüntümüzü siyah-beyaz renklere çevirdik.

In [ ]:
kopya = cv2.cvtColor(kopya, cv2.COLOR_RGB2GRAY)

In [ ]:
plt.imshow(kopya, cmap='gray')

#### Görüntü siyah beyaz olduğu için RGB kanallarını kaybederek (400,400) boyutuna indirgendi.

In [ ]:
kopya.shape

#### Threshold'un daha başarılı uygulanabilmesi için bir miktar Blur uyguluyoruz.

In [ ]:
blur = cv2.GaussianBlur(kopya,(5,5),0)

In [ ]:
plt.imshow(blur,cmap='gray')

Burada görüntü üzerinde  **10 değerinin üzerinde bulduğu tüm değerleri 255 değerine eşitledik**

In [ ]:
thresh = cv2.threshold(blur,10,255, cv2.THRESH_BINARY)[1]

In [ ]:
plt.imshow(thresh, cmap='gray')

Kırpma işlemini yapmak için elimizdeki yeni görüntünün **kenar koordinatlarını(kontur)** buluyoruz.

In [ ]:
kontur = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
# kontur

In [ ]:
 kontur = kontur[0][0]

In [ ]:
kontur.shape

In [ ]:
kontur = kontur[:,0,:]

In [ ]:
kontur.shape

In [ ]:
# kontur

In [ ]:
kontur[:,0].argmax()

In [ ]:
kontur[335]

In [ ]:
kontur[:,0].argmin()

In [ ]:
kontur[111]

In [ ]:
sol = tuple(kontur[kontur[:,0].argmin()])
sağ = tuple(kontur[kontur[:,0].argmax()])
üst = tuple(kontur[kontur[:,1].argmin()])
alt = tuple(kontur[kontur[:,1].argmax()])

Görüntümüzün **4 uç noktasına ait koordinatları**nı elde ettik.

In [ ]:
sol, sağ, üst, alt

In [ ]:
x1 = sol[0]
y1 = üst[1]
x2 = sağ[0]
y2 = alt[1]

In [ ]:
x1, y1, x2, y2

In [ ]:
orijinal = img_list[4].copy()

In [ ]:
plt.imshow(orijinal)

In [ ]:
crop_ilk = orijinal[y1:y2 , x1:x2]

In [ ]:
plt.imshow(crop_ilk)

In [ ]:
crop_ilk.shape

Kıprma işlemini yaptıktan sonra görüntü boyut kayması yaşadığı için tekrardan (400,400) boyutuna eşitliyoruz.

In [ ]:
crop_ilk = cv2.resize(crop_ilk,(400,400))

In [ ]:
plt.imshow(crop_ilk)

Kenarlardaki veriler gereksiz olduğu için bir eşik belirleyip, bir miktar daha görüntüleri kırpıyoruz.

In [ ]:
x = int(x2-x1)*4//100
y = int(y2-y1)*5//100

In [ ]:
x,y

In [ ]:
crop_son = orijinal[y1+y : y2-y , x1+x : x2-x]

In [ ]:
plt.imshow(crop_son)

In [ ]:
crop_son = cv2.resize(crop_son,(400,400))

In [ ]:
plt.imshow(crop_son)

## CLAHE - Kontrast Limitli Adaptif Histogram Eşitleme

CLAHE uygulamak için renk kanalını RGB'den LAB'a çeviriyoruz.

In [ ]:
lab = cv2.cvtColor(crop_son, cv2.COLOR_RGB2LAB)

In [ ]:
lab.shape

In [ ]:
l,a,b = cv2.split(lab)

LAB'daki ''l'' görüntünün siyah-beyaz parlaklık değerini içeriyor. CLAHE işlemini sadece bu katmana uygulayacağız. 

In [ ]:
plt.imshow(l, cmap='gray')

In [ ]:
l.shape

In [ ]:
düz = l.flatten()

In [ ]:
düz.shape

In [ ]:
plt.hist(düz,25,[0,256], color = 'r')
plt.show()

In [ ]:
clahe = cv2.createCLAHE(clipLimit=7.0,tileGridSize=((8,8)))
cl = clahe.apply(l)

In [ ]:
plt.hist(cl.flatten(),25,[0,256], color = 'r')
plt.show()

## CLAHE uygulanmış hali

In [ ]:
plt.imshow(cl)

## CLAHE uygulanmamış hali


In [ ]:
plt.imshow(l)

CLAHE işlemi uyguladığımız katmanı, diğer katmanlarla birleştirip tekrardan görüntümüzü RGB yapıyoruz.

In [ ]:
limg = cv2.merge((cl,a,b))

In [ ]:
son = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

In [ ]:
plt.imshow(son)

In [ ]:
plt.imshow(crop_son)

## Median Blur

In [ ]:
med_son = cv2.medianBlur(son, 3)

In [ ]:
plt.imshow(med_son)

In [ ]:
arka_plan = cv2.medianBlur(son, 37)

In [ ]:
plt.imshow(arka_plan)

In [ ]:
maske = cv2.addWeighted(med_son,1,arka_plan,-1,255)
plt.imshow(maske)

In [ ]:
son_img = cv2.bitwise_and(maske,med_son)

In [ ]:
plt.imshow(son_img)

In [ ]:
plt.imshow(med_son)

#### Yaptığımız tüm işlemleri tek bir for döngüsünde toplayıp, elimizdeki bütün görüntüleri bu döngüyle img_list ismindeki listeye kaydediyoruz.

In [ ]:
img_list = []

from tqdm.notebook import tqdm

for i in tqdm(files):
    image = cv2.imread(path + 'train_img\\'+i)
    image = cv2.resize(image,(400,400))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    kopya = image.copy()
    kopya = cv2.cvtColor(kopya, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(kopya,(5,5),0)
    thresh = cv2.threshold(blur,10,255, cv2.THRESH_BINARY)[1]
    kontur = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    kontur = kontur[0][0]
    kontur = kontur[:,0,:]
    x1 = tuple(kontur[kontur[:,0].argmin()])[0]
    y1 = tuple(kontur[kontur[:,1].argmin()])[1]
    x2 = tuple(kontur[kontur[:,0].argmax()])[0]
    y2 = tuple(kontur[kontur[:,1].argmax()])[1]
    x = int(x2-x1)*4//50
    y = int(y2-y1)*5//50
    kopya2 = image.copy()
    if x2-x1 >100 and y2-y1> 100:
        kopya2 = kopya2[y1+y : y2-y , x1+x : x2-x]
        kopya2 = cv2.resize(kopya2,(400,400))
    lab = cv2.cvtColor(kopya2, cv2.COLOR_RGB2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=5.0,tileGridSize=((8,8)))
    cl = clahe.apply(l)
    limg = cv2.merge((cl,a,b))
    son = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
    med_son = cv2.medianBlur(son, 3)
    arka_plan = cv2.medianBlur(son, 37)
    maske = cv2.addWeighted(med_son,1,arka_plan,-1,255)
    son_img = cv2.bitwise_and(maske,med_son)
    img_list.append(son_img)

In [ ]:
plt.imshow(img_list[6])

In [ ]:
plt.imshow(img_list[6])

In [ ]:
fig = plt.figure(figsize=(20,12))

for i in range(12):
    img = img_list[i]
    fig.add_subplot(3,4,i+1)
    plt.imshow(img)

plt.tight_layout()

In [ ]:
# df['diagnosis']

### One Hot Encoding

In [ ]:
y_train = pd.get_dummies(df['diagnosis']).values

In [ ]:
# y_train

In [ ]:
df['diagnosis'][1]

In [ ]:
y_train[1]

In [ ]:
y_train_son = np.ones(y_train.shape, dtype='uint8')

In [ ]:
# y_train_son

In [ ]:
y_train_son[:,4] = y_train[:,4]

In [ ]:
# y_train_son

In [ ]:
import numpy as np

In [ ]:
np.logical_or(0,0)

In [ ]:
np.logical_or(1,0)

In [ ]:
np.logical_or(0,1)

In [ ]:
np.logical_or(1,1)

In [ ]:
np.logical_and(0,1)

In [ ]:
np.logical_and(1,1)

In [ ]:
for i in range(3,-1,-1):
    y_train_son[:,i] = np.logical_or(y_train[:,i], y_train_son[:,i+1])

In [ ]:
# y_train_son

In [ ]:
# y_train

In [ ]:
x_train = np.array(img_list)

In [ ]:
x_train.shape

In [ ]:
y_train_son.shape

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_val , y_train, y_val = train_test_split(x_train,
                                                   y_train_son,
                                                   test_size=0.15,
                                                   random_state=2019,
                                                   shuffle=True)

In [ ]:
x_train.shape, x_val.shape , y_train.shape, y_val.shape

In [ ]:
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(horizontal_flip=True,vertical_flip=True)
data_generator = datagen.flow(x_train,y_train,batch_size=2,seed=2020)

In [ ]:

from tensorflow.keras.applications import EfficientNetB5


In [ ]:
örnek_model = EfficientNetB5()

In [ ]:
#örnek_model.summary()

In [ ]:
örnek_model2 = EfficientNetB5(include_top=False)
#örnek_model2.summary()

In [ ]:
from keras.models import Sequential
from keras import layers

model = Sequential()
model.add(EfficientNetB5(weights='imagenet',include_top=False, input_shape=(400,400,3)))
model.add(layers.GlobalAveragePooling2D())
model.add(layers.Dropout(0.5))
model.add(layers.Dense(5, activation='sigmoid'))


In [ ]:
from keras.optimizers import Adam
model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.00005), metrics=['accuracy'])


In [ ]:
from keras.callbacks import ReduceLROnPlateau

lr = ReduceLROnPlateau(monitor = 'val_loss',
                      patience = 3,
                      verbose = 1,
                      mode='auto',
                      factor=0.25,
                      min_lr=0.000001)

In [ ]:
history = model.fit(data_generator,
                             steps_per_epoch = 1000,
                             epochs = 30,
                             validation_data = (x_val,y_val),
                             callbacks = [lr])


In [ ]:
# History'den eğitim ve doğrulama metriklerini alma
history_dict = history.history

In [ ]:
# Eğitim ve doğrulama kaybı (loss)
loss_values = history_dict['loss']
val_loss_values = history_dict['val_loss']

In [ ]:
# Eğitim ve doğrulama doğruluğu (accuracy)
acc_values = history_dict['accuracy']
val_acc_values = history_dict['val_accuracy']

In [ ]:
epochs = range(1, len(loss_values) + 1)

In [ ]:
# Kayıp (Loss) Grafiği
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(epochs, loss_values, 'bo', label='Training Loss')  # 'bo' mavi noktalar
plt.plot(epochs, val_loss_values, 'b', label='Validation Loss')  # 'b' mavi çizgi
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

In [ ]:
# Doğruluk (Accuracy) Grafiği
plt.subplot(1, 2, 2)
plt.plot(epochs, acc_values, 'ro', label='Training Accuracy')  # 'ro' kırmızı noktalar
plt.plot(epochs, val_acc_values, 'r', label='Validation Accuracy')  # 'r' kırmızı çizgi
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()